# 🎨 Zero-Trust Ephemeral Ciphertext Queue & Media Studio Worker (`v1.3.0`)

Execute image & video generation (`MOSS-Image-SFT`, `SDXL`, `FLUX.1`, `Wan 2.1/2.2 T2V & I2V`, `Wan 2.2 TI2V-5B`, `HunyuanVideo-1.5`) on untrusted remote GPUs with zero plaintext persistence:
1. Run **Cell 1** once to enter your session passphrase in memory (0.1s prompt before background package setup), apply selective `KISSSHOT_` IPython history scrubbing (preserving unrelated notebook work), verify optional runtime lock (`strict=False` compatibility mode by default), and initialize the pinned-commit TorchAO FP8/BF16 & Real-ESRGAN 2x engines.
2. Paste `KISSSHOT_TASK_HEX` from your local `kiss-shot panel` into **Cell 2**:
   - **Automated Queue Daemon v2 (Recommended)**: Paste the daemon token once (`🛰️ 复制远端自动接单守护口令`). The daemon claims `_queue/v2/` (and legacy `_queue/`) jobs with `_claims/<job_id>.json` lease heartbeats, strips `hf_token` from queue payloads, supports dynamic LoRA strength (`_set_lora_strength`), and atomically commits `_inbox/pkt_*.bin` + `_receipts/<job_id>.json` while cleaning `_queue/`/`_claims/`/`_staging/`.
   - **Single-Shot Mode**: Paste a one-off task token (`🔒 手动模式`) to run a single inference job.
3. Outputs are packed purely in memory (`BytesIO` / `/dev/shm`) into a double-AEAD archive (`AES-256-GCM` outer `AAD_ARCHIVE` envelope + `256-bit WZ_AES` `ZIP_STORED` inner archive with recipe metadata) and **conditionally burned only after local verified decryption**.

In [ ]:
# === Cell 1: Passphrase Unlock, Selective History Scrubbing, Runtime Lock, Pinned Supply-Chain & Double-AEAD Engine ===
import subprocess, sys, os, io, gc, json, re, math, hashlib, hmac, inspect, shutil, sqlite3, time, secrets, tempfile, threading, importlib.util, getpass
import importlib.metadata as importlib_metadata

# Prompt immediately at 0.1s before installing packages so window focus is never lost
SESSION_PASSPHRASE = getpass.getpass("🔑 Enter your Kiss-Shot session passphrase (in-memory only): ").strip()
assert SESSION_PASSPHRASE, "Passphrase cannot be empty."
print("Passphrase locked in memory. Installing/verifying inference & cryptography dependencies...")

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

_SENSITIVE_HISTORY_PATTERNS = (
    "%KISSSHOT_%",
    "%SESSION_PASSPHRASE%",
    "%_TASK_HEX%",
    "%_LLM_HEX%",
)
_SENSITIVE_MEM_TOKENS = ("KISSSHOT_", "SESSION_PASSPHRASE", "_TASK_HEX", "_LLM_HEX")

def _scrub_sqlite_history_file(db_path: str) -> int:
    """K12: Selectively delete only KISSSHOT_-related sensitive rows; never wipe unrelated user work."""
    if not db_path or not os.path.exists(db_path):
        return 0
    removed = 0
    con = sqlite3.connect(db_path)
    try:
        cols = {row[1] for row in con.execute("PRAGMA table_info(history)").fetchall()}
        clauses, params = [], []
        for pat in _SENSITIVE_HISTORY_PATTERNS:
            if "source" in cols:
                clauses.append("source LIKE ?")
                params.append(pat)
            if "source_raw" in cols:
                clauses.append("source_raw LIKE ?")
                params.append(pat)
        if clauses:
            cur = con.execute(f"DELETE FROM history WHERE {' OR '.join(clauses)}", params)
            removed += max(0, int(cur.rowcount or 0))
        tables = {row[0] for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
        if "output_history" in tables:
            out_cols = {row[1] for row in con.execute("PRAGMA table_info(output_history)").fetchall()}
            if "output" in out_cols:
                o_clauses = ["output LIKE ?" for _ in _SENSITIVE_HISTORY_PATTERNS]
                con.execute(f"DELETE FROM output_history WHERE {' OR '.join(o_clauses)}", list(_SENSITIVE_HISTORY_PATTERNS))
        con.commit()
        if removed > 0:
            con.execute("VACUUM")
    finally:
        con.close()
    return removed

def scrub_ipython_history(hist_db_path: str | None = None) -> int:
    """K12: Redact in-memory KISSSHOT_ entries and selectively delete only KISSSHOT_ rows from SQLite history."""
    removed = 0
    if hist_db_path is not None:
        return _scrub_sqlite_history_file(hist_db_path)
    try:
        ip = get_ipython()  # noqa: F821
        if ip and getattr(ip, "history_manager", None):
            hm = ip.history_manager
            hm.enabled = False
            if getattr(hm, "db", None):
                try:
                    cols = {row[1] for row in hm.db.execute("PRAGMA table_info(history)").fetchall()}
                    clauses, params = [], []
                    for pat in _SENSITIVE_HISTORY_PATTERNS:
                        if "source" in cols:
                            clauses.append("source LIKE ?")
                            params.append(pat)
                        if "source_raw" in cols:
                            clauses.append("source_raw LIKE ?")
                            params.append(pat)
                    if clauses:
                        cur = hm.db.execute(f"DELETE FROM history WHERE {' OR '.join(clauses)}", params)
                        removed += max(0, int(cur.rowcount or 0))
                        hm.db.commit()
                except Exception:
                    pass
        if ip and hasattr(ip, "user_ns"):
            ns = ip.user_ns
            for k in ("In", "_ih"):
                arr = ns.get(k)
                if isinstance(arr, list):
                    for idx in range(len(arr)):
                        text = str(arr[idx])
                        if any(tok in text for tok in _SENSITIVE_MEM_TOKENS):
                            arr[idx] = "# [cleared]"
            for out_k in ("Out", "_oh"):
                od = ns.get(out_k)
                if isinstance(od, dict):
                    for k in list(od.keys()):
                        text = str(od[k])
                        if any(tok in text for tok in _SENSITIVE_MEM_TOKENS):
                            od.pop(k, None)
    except Exception:
        pass
    hist_db = os.path.expanduser("~/.ipython/profile_default/history.sqlite")
    try:
        removed += _scrub_sqlite_history_file(hist_db)
    except Exception:
        pass
    return removed

scrub_ipython_history()

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "diffusers>=0.35.0", "transformers>=4.49.0", "accelerate>=1.4.0", "sentencepiece",
    "ftfy", "peft", "spandrel", "imageio[ffmpeg]", "cryptography", "pyzipper",
    "huggingface_hub", "hf_transfer", "bitsandbytes"
])
if importlib.util.find_spec("torchao") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "torchao"])
    except Exception:
        pass

RUNTIME_LOCK_PATH = os.environ.get("KISSSHOT_RUNTIME_LOCK", "").strip()
RUNTIME_LOCK_STRICT = False

try:
    from kiss_shot.runtime_lock import verify_runtime_lock  # type: ignore
except ImportError:
    _RUNTIME_LOCK_SCHEMA = "kissshot-runtime-lock-v1"

    def _probe_pkg_ver(pkg_name: str) -> str | None:
        for cand in (pkg_name, pkg_name.replace("_", "-"), pkg_name.replace("-", "_")):
            try:
                return importlib_metadata.version(cand)
            except importlib_metadata.PackageNotFoundError:
                continue
        return None

    def verify_runtime_lock(
        manifest: dict | None = None,
        *,
        profile: str | None = None,
        strict: bool = False,
        installed_versions: dict | None = None,
    ) -> dict:
        if not manifest:
            if strict:
                raise RuntimeError("Missing runtime lock manifest while strict=True")
            return {"ok": True, "allowed": True, "mode": "unlocked", "warnings": [], "missing": [], "mismatches": []}
        warnings, missing, mismatches = [], [], []
        schema = str(manifest.get("schema", ""))
        if schema != _RUNTIME_LOCK_SCHEMA:
            warnings.append(f"Unexpected schema: {schema!r}")
        m_profile = str(manifest.get("profile", ""))
        if profile and m_profile and m_profile != profile:
            warnings.append(f"Profile mismatch: expected {profile!r}, got {m_profile!r}")
        packages = manifest.get("packages")
        if not isinstance(packages, dict):
            raise ValueError("Runtime lock manifest 'packages' must be a dictionary")
        declared = str(manifest.get("manifest_sha256", "")).strip().lower()
        if declared:
            canon = json.dumps(
                {"schema": schema, "profile": m_profile, "packages": {k: packages[k] for k in sorted(packages)}},
                ensure_ascii=False, sort_keys=True, separators=(",", ":"),
            ).encode("utf-8")
            if not hmac.compare_digest(declared, hashlib.sha256(canon).hexdigest()):
                warnings.append("Runtime lock manifest_sha256 integrity check failed")
        for pkg_name, entry in packages.items():
            if not isinstance(entry, dict) or not entry.get("installed", True):
                continue
            locked_ver = str(entry.get("version", "")).strip()
            if not locked_ver:
                continue
            actual_ver = installed_versions[pkg_name] if (installed_versions is not None and pkg_name in installed_versions) else _probe_pkg_ver(pkg_name)
            if not actual_ver:
                missing.append(pkg_name)
            elif actual_ver != locked_ver:
                mismatches.append({"package": pkg_name, "expected": locked_ver, "actual": actual_ver})
        clean = not warnings and not missing and not mismatches
        if not clean and strict:
            raise RuntimeError(f"Runtime lock verification failed (warnings={warnings}, missing={missing}, mismatches={mismatches})")
        return {"ok": clean, "allowed": True if (clean or not strict) else False, "mode": "verified" if clean else "warn", "warnings": warnings, "missing": missing, "mismatches": mismatches}

_lock_manifest = json.loads(open(RUNTIME_LOCK_PATH, encoding="utf-8").read()) if (RUNTIME_LOCK_PATH and os.path.isfile(RUNTIME_LOCK_PATH)) else None
_lock_result = verify_runtime_lock(_lock_manifest, profile="studio", strict=RUNTIME_LOCK_STRICT)
print(f"🔒 Runtime lock check: mode={_lock_result['mode']} (ok={_lock_result['ok']})")

import numpy as np
import torch, pyzipper
from PIL import Image, ImageFilter
from huggingface_hub import CommitOperationAdd, CommitOperationDelete, HfApi, hf_hub_download, snapshot_download
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    gpu_name = torch.cuda.get_device_name(0)
    cuda_cc = torch.cuda.get_device_capability(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"✅ Worker ready on GPU: {gpu_name} ({vram_gb} GB VRAM, SM {cuda_cc[0]}.{cuda_cc[1]})")
else:
    cuda_cc, vram_gb = (0, 0), 0.0
    print("⚠️ Running on CPU; attach a CUDA GPU runtime for accelerated generation.")

KDF_ITERATIONS = 210_000
SALT_BYTES, NONCE_BYTES, TAG_BYTES = 16, 12, 16
AAD_PAYLOAD = b"kissshot-payload-v2"
AAD_QUEUE = b"kissshot-queue-v2"
AAD_BLOB = b"kissshot-blob-v2"
AAD_ARCHIVE = b"kissshot-archive-v2"
MAX_TTL_SECONDS = 86_400

_SAFE_STAGING_RE = re.compile(r"^_staging/(?:ref|mask)_[0-9a-f]{12}\.bin$")
_SAFE_QUEUE_V2_RE = re.compile(r"^_queue/v2/job_[0-9]{9,13}_[A-Za-z0-9_-]{6,48}\.bin$")
_SAFE_QUEUE_LEGACY_RE = re.compile(r"^_queue/job_[0-9]{9,13}_[0-9a-f]{8}\.bin$")
_COMMIT_SHA_RE = re.compile(r"^[0-9a-f]{40}$")
_HF_REPO_ID_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]*/[A-Za-z0-9][A-Za-z0-9._-]*$")

# K02: Per-repository pinned commit revisions and verified SHA-256 digests
PINNED_MODEL_REVISIONS: dict[str, str] = {
    "ai-forever/Real-ESRGAN": "8110204ebf8d25c031b66c26c2d1098aa831157e",
    "OpenMOSS-Team/MOSS-Image-SFT": "f972bd4b4b9e3852133cc5da954d87bc32cd8fb9",
    "stabilityai/stable-diffusion-xl-base-1.0": "462165984030d82259a11f4367a4eed129e94a7b",
    "black-forest-labs/FLUX.1-schnell": "169573f54945f633efbe126a40a5676b4228363c",
    "Wan-AI/Wan2.1-T2V-1.3B-Diffusers": "0fad780a534b6463e45facd96134c9f345acfa5b",
    "Wan-AI/Wan2.2-TI2V-5B-Diffusers": "b8fff7315c768468a5333511427288870b2e9635",
    "hunyuanvideo-community/HunyuanVideo-1.5-Diffusers-720p_t2v": "f4dbc4a1efa4ac8ea56680cdf79d9f455105e814",
    "Wan-AI/Wan2.2-I2V-A14B-Diffusers": "596658fd9ca6b7b71d5057529bbf319ecbc61d74",
}
PINNED_FILE_SHA256: dict[tuple[str, str], str] = {
    ("ai-forever/Real-ESRGAN", "RealESRGAN_x2.pth"): "c830d067d54fc767b9543a8432f36d91bc2de313584e8bbfe4ac26a47339e899",
    ("OpenMOSS-Team/MOSS-Image-SFT", "pipeline_moss_image.py"): "1dac1aca9feb1bfdfc727a401e4b67d61a505d3190706d1b7d2d784f88678b31",
}

# K08: Strict allowlist for custom MOSS pipeline Python code + executable stripping for all model repos
APPROVED_MOSS_SOURCES: dict[str, dict[str, str]] = {
    "OpenMOSS-Team/MOSS-Image-SFT": {
        "revision": "f972bd4b4b9e3852133cc5da954d87bc32cd8fb9",
        "filename": "pipeline_moss_image.py",
        "sha256": "1dac1aca9feb1bfdfc727a401e4b67d61a505d3190706d1b7d2d784f88678b31",
    },
}
_EXECUTABLE_SUFFIXES = (
    ".py", ".pyc", ".pyo", ".pyd", ".so", ".dll", ".dylib",
    ".exe", ".sh", ".bash", ".bat", ".cmd", ".ps1", ".js", ".mjs", ".cjs",
)
_EXECUTABLE_IGNORE_PATTERNS = [f"*{ext}" for ext in _EXECUTABLE_SUFFIXES]

_PIPE_CACHE = {"sig": None, "pipe": None, "upscaler": None, "lora_sig": None, "lora_adapter": "ks_lora"}
_TORCHAO_CAPS = {"checked": False, "dyn_fp8": None, "wo_cfg": None}
_ENSURED_REPOS: set[str] = set()

def _sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest().lower()

def _job_arch(task_or_arch) -> str:
    """K01: Normalize architecture identifier while preserving wan_ti2v_5b (backward-compatible with wan_ti2v)."""
    raw = task_or_arch.get("arch", "") if isinstance(task_or_arch, dict) else str(task_or_arch or "")
    arch = str(raw).strip().lower()
    if arch in ("wan_ti2v", "wan_ti2v_5b"):
        return "wan_ti2v_5b"
    return arch

def _is_executable_repo_file(rel_path: str) -> bool:
    low = str(rel_path or "").strip().lower()
    return any(low.endswith(ext) for ext in _EXECUTABLE_SUFFIXES)

def _strip_repo_executables(root_dir: str) -> list[str]:
    """K08: Remove any executable or script files from downloaded model repository directory."""
    removed: list[str] = []
    if not os.path.isdir(root_dir):
        return removed
    for cur_root, _dirs, files in os.walk(root_dir):
        for fn in files:
            if _is_executable_repo_file(fn):
                fp = os.path.join(cur_root, fn)
                try:
                    os.remove(fp)
                    removed.append(os.path.relpath(fp, root_dir).replace("\\", "/"))
                except OSError:
                    pass
    return removed

def verified_model_file(
    repo_id: str,
    filename: str,
    *,
    local_dir: str,
    revision: str | None = None,
    expected_sha256: str = "",
    token=False,
    pinned_revisions: dict | None = None,
    pinned_hashes: dict | None = None,
    downloader=None,
) -> str:
    """K02: Download a single model file strictly from a pinned per-repo commit SHA and verify its SHA-256."""
    rev_map = pinned_revisions if pinned_revisions is not None else PINNED_MODEL_REVISIONS
    hash_map = pinned_hashes if pinned_hashes is not None else PINNED_FILE_SHA256
    rev = str(revision or rev_map.get(repo_id) or "").strip().lower()
    if not rev or not _COMMIT_SHA_RE.fullmatch(rev):
        raise ValueError(f"Unpinned model revision for {repo_id!r}: expected 40-char commit SHA, got {rev!r}")
    os.makedirs(local_dir, exist_ok=True)
    dl = downloader or hf_hub_download
    local_path = dl(repo_id=repo_id, filename=filename, revision=rev, local_dir=local_dir, token=token)
    exp = str(expected_sha256 or hash_map.get((repo_id, filename)) or "").strip().lower()
    if exp:
        actual = _sha256_file(local_path)
        if not hmac.compare_digest(actual, exp):
            try:
                os.remove(local_path)
            except OSError:
                pass
            raise RuntimeError(
                f"SHA-256 verification failed for {repo_id}/{filename}@{rev[:12]}: expected {exp[:16]}..., got {actual[:16]}..."
            )
    return local_path

def _approved_moss_source(
    target_repo: str,
    local_dir: str,
    *,
    revision: str | None = None,
    token=False,
    approved_sources: dict | None = None,
    downloader=None,
) -> str:
    """K08: Only allow pinned, SHA-256-verified MOSS pipeline code from explicitly approved repositories."""
    registry = approved_sources if approved_sources is not None else APPROVED_MOSS_SOURCES
    spec = registry.get(target_repo)
    if not spec:
        raise PermissionError(f"Unapproved MOSS code repository rejected: {target_repo!r}")
    pinned_rev = str(spec["revision"]).strip().lower()
    if revision and str(revision).strip().lower() != pinned_rev:
        raise PermissionError(f"MOSS revision mismatch for {target_repo!r}: expected {pinned_rev}, got {revision!r}")
    return verified_model_file(
        repo_id=target_repo,
        filename=spec["filename"],
        local_dir=local_dir,
        revision=pinned_rev,
        expected_sha256=spec["sha256"],
        token=token,
        pinned_revisions={target_repo: pinned_rev},
        pinned_hashes={(target_repo, spec["filename"]): spec["sha256"]},
        downloader=downloader,
    )

def _set_lora_strength(
    pipe,
    lora_repo: str = "",
    lora_weight: str = "",
    lora_strength: float = 0.8,
    *,
    hf_token: str = "",
    adapter_name: str = "ks_lora",
    current_state: dict | None = None,
) -> dict:
    """K15: Safely load, scale, or unload LoRA weights without leaking stale adapters across cached pipeline runs."""
    state = current_state if current_state is not None else _PIPE_CACHE
    repo = str(lora_repo or "").strip()
    weight = str(lora_weight or "").strip()
    strength = max(0.0, min(2.0, float(lora_strength if lora_strength is not None else 0.8)))

    if not repo or strength <= 0.0:
        if state.get("lora_sig") is not None:
            if hasattr(pipe, "unload_lora_weights"):
                pipe.unload_lora_weights()
            elif hasattr(pipe, "disable_lora"):
                pipe.disable_lora()
            elif hasattr(pipe, "set_adapters"):
                pipe.set_adapters([], adapter_weights=[])
            state["lora_sig"] = None
        return {"active": False, "repo": "", "weight": "", "strength": 0.0, "scale_kwarg": False}

    if not _HF_REPO_ID_RE.fullmatch(repo) or ".." in repo:
        raise ValueError(f"Invalid LoRA repository ID: {repo!r}")
    if weight and ("\\" in weight or ".." in weight or weight.startswith("/")):
        raise ValueError(f"Invalid LoRA weight filename: {weight!r}")
    if not hasattr(pipe, "load_lora_weights"):
        raise ValueError("Current pipeline does not support dynamic LoRA weights.")

    desired_sig = (repo, weight)
    if state.get("lora_sig") != desired_sig:
        if state.get("lora_sig") is not None and hasattr(pipe, "unload_lora_weights"):
            pipe.unload_lora_weights()
        load_sig = set(inspect.signature(pipe.load_lora_weights).parameters.keys())
        base_kw: dict = {}
        if weight:
            base_kw["weight_name"] = weight
        if "adapter_name" in load_sig:
            base_kw["adapter_name"] = adapter_name
        try:
            kw_pub = dict(base_kw)
            if "token" in load_sig:
                kw_pub["token"] = False
            pipe.load_lora_weights(repo, **kw_pub)
        except Exception:
            if not hf_token:
                raise
            kw_tok = dict(base_kw)
            if "token" in load_sig:
                kw_tok["token"] = hf_token
            pipe.load_lora_weights(repo, **kw_tok)
        state["lora_sig"] = desired_sig
        state["lora_adapter"] = adapter_name

    used_scale_kwarg = False
    if hasattr(pipe, "set_adapters"):
        pipe.set_adapters([adapter_name], adapter_weights=[strength])
    elif hasattr(pipe, "fuse_lora"):
        pipe.fuse_lora(lora_scale=strength)
    else:
        used_scale_kwarg = True
    return {
        "active": True,
        "repo": repo,
        "weight": weight,
        "strength": strength,
        "adapter_name": adapter_name,
        "scale_kwarg": used_scale_kwarg,
    }

def derive_key(pwd: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=KDF_ITERATIONS)
    return kdf.derive(pwd.encode("utf-8"))

def decrypt_payload_hex(hex_str: str, pwd: str, aad: bytes = AAD_PAYLOAD) -> dict:
    raw = bytes.fromhex("".join(hex_str.split()))
    if len(raw) <= SALT_BYTES + NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated ciphertext")
    pt = AESGCM(derive_key(pwd, raw[:SALT_BYTES])).decrypt(
        raw[SALT_BYTES:SALT_BYTES + NONCE_BYTES],
        raw[SALT_BYTES + NONCE_BYTES:],
        aad,
    )
    parsed = json.loads(pt.decode("utf-8"))
    if isinstance(parsed, dict) and parsed.get("_ks_v") == 2 and "d" in parsed:
        iat = int(parsed.get("_ks_iat", 0) or 0)
        if not iat or abs(time.time() - iat) > MAX_TTL_SECONDS:
            raise ValueError("Expired task payload (>24h)")
        return parsed["d"]
    return parsed

def encrypt_bytes_aes(raw: bytes, pwd: str, aad: bytes = AAD_BLOB) -> bytes:
    salt, nonce = secrets.token_bytes(SALT_BYTES), secrets.token_bytes(NONCE_BYTES)
    ct = AESGCM(derive_key(pwd, salt)).encrypt(nonce, raw, aad)
    return salt + nonce + ct

def decrypt_bytes_aes(enc: bytes, pwd: str, aad: bytes = AAD_BLOB) -> bytes:
    if len(enc) <= SALT_BYTES + NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated encrypted blob")
    return AESGCM(derive_key(pwd, enc[:SALT_BYTES])).decrypt(
        enc[SALT_BYTES:SALT_BYTES + NONCE_BYTES],
        enc[SALT_BYTES + NONCE_BYTES:],
        aad,
    )

def decrypt_queue_job(enc: bytes, pwd: str) -> dict:
    pt = decrypt_bytes_aes(enc, pwd, aad=AAD_QUEUE)
    parsed = json.loads(pt.decode("utf-8"))
    if not isinstance(parsed, dict) or parsed.get("_ks_v") != 2 or "d" not in parsed:
        raise ValueError("Malformed queue job envelope")
    iat = int(parsed.get("_ks_iat", 0) or 0)
    if not iat or abs(time.time() - iat) > MAX_TTL_SECONDS:
        raise ValueError("Expired queue job (>24h)")
    return parsed["d"]

def pack_encrypted_zip(items: list, pwd: str, meta: dict) -> bytes:
    buf = io.BytesIO()
    pwd_b = pwd.encode("utf-8")
    manifest = {"version": 2, "created_at": int(time.time()), "files": [], "meta": meta}
    with pyzipper.AESZipFile(buf, "w", compression=pyzipper.ZIP_STORED, encryption=pyzipper.WZ_AES) as zf:
        zf.setpassword(pwd_b)
        zf.setencryption(pyzipper.WZ_AES, nbits=256)
        for i, (ext, data) in enumerate(items):
            inner = f"chunk_{i:03d}.dat"
            zf.writestr(inner, data)
            manifest["files"].append({"inner": inner, "ext": ext.lstrip(".") or "bin"})
        zf.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False).encode("utf-8"))
    return encrypt_bytes_aes(buf.getvalue(), pwd, aad=AAD_ARCHIVE)

def _get_torchao_caps() -> dict:
    if _TORCHAO_CAPS["checked"] or not torch.cuda.is_available():
        return _TORCHAO_CAPS
    _TORCHAO_CAPS["checked"] = True
    try:
        import torchao.quantization as tq
        quantize_ = tq.quantize_
        if cuda_cc >= (8, 9) and hasattr(tq, "Float8DynamicActivationFloat8WeightConfig"):
            try:
                probe = torch.nn.Linear(1024, 1024, bias=False, device="cuda", dtype=torch.bfloat16)
                cfg_dyn = tq.Float8DynamicActivationFloat8WeightConfig()
                quantize_(probe, cfg_dyn)
                out = probe(torch.ones((1, 32, 1024), device="cuda", dtype=torch.bfloat16))
                if out.shape == (1, 32, 1024):
                    _TORCHAO_CAPS["dyn_fp8"] = cfg_dyn
                del probe, out
            except Exception:
                pass
        for cls_name in ("Float8WeightOnlyConfig", "Int8WeightOnlyConfig"):
            cfg_cls = getattr(tq, cls_name, None)
            if cfg_cls is None:
                continue
            try:
                probe = torch.nn.Linear(1024, 1024, bias=True, device="cuda", dtype=torch.bfloat16)
                cfg_wo = cfg_cls()
                quantize_(probe, cfg_wo)
                out = probe(torch.ones((1, 32, 1024), device="cuda", dtype=torch.bfloat16))
                if out.shape == (1, 32, 1024):
                    _TORCHAO_CAPS["wo_cfg"] = cfg_wo
                    del probe, out
                    break
                del probe, out
            except Exception:
                pass
        torch.cuda.empty_cache()
    except Exception:
        pass
    return _TORCHAO_CAPS

def _dit_linear_filter(mod: torch.nn.Module, _fqn: str) -> bool:
    return (
        isinstance(mod, torch.nn.Linear)
        and mod.in_features >= 1024
        and mod.out_features >= 1024
        and (mod.in_features % 16 == 0)
        and (mod.out_features % 16 == 0)
    )

def _stream_dit_to_cuda(transformer: torch.nn.Module, allow_fp8_tc: bool = True, need_shrink: bool = False) -> str:
    caps = _get_torchao_caps()
    q_cfg = None
    mode_tag = "BF16 full-VRAM"
    if allow_fp8_tc and caps["dyn_fp8"] is not None:
        q_cfg = caps["dyn_fp8"]
        mode_tag = "FP8 TensorCore full-VRAM"
    elif need_shrink and caps["wo_cfg"] is not None:
        q_cfg = caps["wo_cfg"]
        mode_tag = "8-bit weight-only full-VRAM"
    if q_cfg is not None:
        from torchao.quantization import quantize_
        hooked = False
        for attr in ("layers", "transformer_blocks", "single_transformer_blocks"):
            blks = getattr(transformer, attr, None)
            if isinstance(blks, torch.nn.ModuleList):
                for blk in blks:
                    blk.to("cuda")
                    quantize_(blk, q_cfg, filter_fn=_dit_linear_filter)
                hooked = True
        transformer.to("cuda")
        if not hooked:
            quantize_(transformer, q_cfg, filter_fn=_dit_linear_filter)
        torch.cuda.empty_cache()
        return mode_tag
    if need_shrink and hasattr(transformer, "enable_layerwise_casting"):
        transformer.enable_layerwise_casting(storage_dtype=torch.float8_e4m3fn, compute_dtype=torch.bfloat16)
        transformer.to("cuda")
        return "FP8 layerwise full-VRAM"
    transformer.to("cuda")
    return mode_tag

def _load_text_encoder_vram(model_cls, load_path: str, subfolder: str | None = None):
    kw = {"local_files_only": True}
    if subfolder:
        kw["subfolder"] = subfolder
    if vram_gb >= 34.0:
        enc = model_cls.from_pretrained(load_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, **kw).eval()
        if hasattr(enc, "visual"):
            enc.visual = None
        return enc.to("cuda")
    caps = _get_torchao_caps()
    if caps["wo_cfg"] is not None:
        from torchao.quantization import quantize_
        enc = model_cls.from_pretrained(load_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, **kw).eval()
        if hasattr(enc, "visual"):
            enc.visual = None
        enc.to("cuda")
        quantize_(enc, caps["wo_cfg"], filter_fn=lambda m, _: isinstance(m, torch.nn.Linear) and m.in_features >= 512)
        torch.cuda.empty_cache()
        return enc
    try:
        from transformers import BitsAndBytesConfig
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=["visual"],
        )
        enc = model_cls.from_pretrained(load_path, quantization_config=bnb_cfg, device_map={"": 0}, low_cpu_mem_usage=True, **kw).eval()
        if hasattr(enc, "visual"):
            enc.visual = None
        return enc
    except Exception:
        enc = model_cls.from_pretrained(load_path, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, **kw).eval()
        if hasattr(enc, "visual"):
            enc.visual = None
        return enc.to("cuda")

def _upscale_pil_2x(img: Image.Image) -> Image.Image:
    from spandrel import ModelLoader
    if _PIPE_CACHE["upscaler"] is None:
        up_dir = "/tmp/kissshot_ckpt/_upscaler"
        os.makedirs(up_dir, exist_ok=True)
        # K02: Pinned commit SHA + SHA-256 integrity verification for Real-ESRGAN
        up_p = verified_model_file(
            repo_id="ai-forever/Real-ESRGAN",
            filename="RealESRGAN_x2.pth",
            local_dir=up_dir,
            token=False,
        )
        shutil.rmtree(os.path.join(up_dir, ".cache"), ignore_errors=True)
        _PIPE_CACHE["upscaler"] = ModelLoader().load_from_file(up_p).model.eval().to("cuda", dtype=torch.float16)
    net = _PIPE_CACHE["upscaler"]
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    ten = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    _, _, h, w = ten.shape
    scale, tile, overlap = 2, 512, 32
    out = torch.zeros((1, 3, h * scale, w * scale), dtype=torch.float32)
    wgt = torch.zeros((1, 1, h * scale, w * scale), dtype=torch.float32)
    with torch.no_grad():
        for y in range(0, h, tile - overlap):
            for x in range(0, w, tile - overlap):
                y1, x1 = min(h, y + tile), min(w, x + tile)
                y0, x0 = max(0, y1 - tile), max(0, x1 - tile)
                patch = ten[:, :, y0:y1, x0:x1].to("cuda", dtype=torch.float16, non_blocking=True)
                sr = net(patch).float().cpu().clamp(0.0, 1.0)
                out[:, :, y0 * scale:y1 * scale, x0 * scale:x1 * scale] += sr
                wgt[:, :, y0 * scale:y1 * scale, x0 * scale:x1 * scale] += 1.0
    res = (out / wgt.clamp_min(1.0)).squeeze(0).permute(1, 2, 0).numpy()
    return Image.fromarray((res * 255.0).round().astype(np.uint8))

def _moss_img2img_or_inpaint(pipe, ref_img: Image.Image, mask_img: Image.Image | None, prompt_str: str, neg_str: str,
                            width: int, height: int, steps: int, cfg_val: float, strength: float, gen: torch.Generator) -> Image.Image:
    from diffusers.utils.torch_utils import randn_tensor
    ref_resized = ref_img.convert("RGB").resize((width, height), Image.Resampling.LANCZOS)
    device = pipe._execution_device
    prompt_embeds, prompt_mask = pipe.encode_prompt(prompt_str, 4096)
    positive, positive_mask = prompt_embeds.to(device), prompt_mask.to(device)
    do_cfg = cfg_val != 1.0
    uncond_skip = () if neg_str else (22,)
    if do_cfg:
        neg_embeds, neg_mask = pipe.encode_prompt([neg_str], 4096)
        negative, negative_mask = neg_embeds.to(device), neg_mask.to(device)

    x_np = (np.asarray(ref_resized, dtype=np.float32) / 127.5) - 1.0
    x_in = torch.from_numpy(x_np).permute(2, 0, 1).unsqueeze(0).to(device=device, dtype=pipe.vae.dtype)
    with torch.no_grad():
        z_raw = pipe.vae.encode(x_in).latent_dist.mode()
        b, c_v, h2, w2 = z_raw.shape
        h_l, w_l = h2 // 2, w2 // 2
        z_pack = z_raw.reshape(b, c_v, h_l, 2, w_l, 2).permute(0, 1, 3, 5, 2, 4).reshape(b, c_v * 4, h_l, w_l)
        bn = pipe.vae.bn
        eps = pipe.vae.config.batch_norm_eps
        z_0 = ((z_pack - bn.running_mean.to(z_pack).view(1, -1, 1, 1)) / (bn.running_var + eps).sqrt().to(z_pack).view(1, -1, 1, 1)).float()

    mask_lat, feather_mask = None, None
    if mask_img is not None:
        m_resized = mask_img.convert("L").resize((width, height), Image.Resampling.BILINEAR)
        feather_mask = m_resized.filter(ImageFilter.GaussianBlur(radius=6))
        m_small = feather_mask.resize((w_l, h_l), Image.Resampling.BILINEAR)
        mask_lat = torch.from_numpy(np.asarray(m_small, dtype=np.float32) / 255.0).view(1, 1, h_l, w_l).to(device=device, dtype=torch.float32)

    pipe.scheduler.set_timesteps(steps, device=device)
    total_t = len(pipe.scheduler.timesteps)
    t_idx = min(total_t - 1, max(0, int(round((1.0 - strength) * (total_t - 1)))))
    pipe.scheduler.timesteps = pipe.scheduler.timesteps[t_idx:]
    pipe.scheduler.sigmas = pipe.scheduler.sigmas[t_idx:]
    pipe.scheduler._step_index = None

    noise = randn_tensor(z_0.shape, generator=gen, device=device, dtype=torch.float32)
    sigma0 = pipe.scheduler.sigmas[0].to(z_0)
    latents = (1.0 - sigma0) * z_0 + sigma0 * noise

    with torch.no_grad():
        for s_i, timestep in enumerate(pipe.progress_bar(pipe.scheduler.timesteps)):
            pred = pipe.transformer(latents, timestep, positive, positive_mask).sample.float()
            if do_cfg:
                sk = {"skip_layers": uncond_skip} if uncond_skip else {}
                uncond = pipe.transformer(latents, timestep, negative, negative_mask, **sk).sample.float()
                pred = uncond + cfg_val * (pred - uncond)
            latents = pipe.scheduler.step(pred, timestep, latents, return_dict=False)[0]
            if mask_lat is not None and (s_i + 1) < len(pipe.scheduler.sigmas):
                sig_next = pipe.scheduler.sigmas[s_i + 1].to(z_0)
                ref_noisy = (1.0 - sig_next) * z_0 + sig_next * noise
                latents = mask_lat * latents + (1.0 - mask_lat) * ref_noisy

        z_out = latents.to(dtype=pipe.vae.dtype)
        z_out = z_out * (bn.running_var + eps).sqrt().to(z_out).view(1, -1, 1, 1) + bn.running_mean.to(z_out).view(1, -1, 1, 1)
        b, c, h_o, w_o = z_out.shape
        z_out = z_out.reshape(b, c // 4, 2, 2, h_o, w_o).permute(0, 1, 4, 2, 5, 3).reshape(b, c // 4, 2 * h_o, 2 * w_o)
        decoded = pipe.vae.decode(z_out, return_dict=False)[0].clamp(-1, 1)
        out_pil = pipe.image_processor.postprocess(decoded, output_type="pil")[0]
    pipe.maybe_free_model_hooks()
    if feather_mask is not None:
        out_pil = Image.composite(out_pil, ref_resized, feather_mask)
    return out_pil

def _pull_staging_img(mailbox_repo: str, staging_rel: str, hf_tok: str, *, burn_now: bool = False) -> Image.Image:
    if not _SAFE_STAGING_RE.fullmatch(staging_rel):
        raise ValueError(f"Unsafe staging path rejected: {staging_rel!r}")
    shm = "/dev/shm" if os.path.isdir("/dev/shm") else None
    with tempfile.TemporaryDirectory(dir=shm) as td:
        enc_p = hf_hub_download(repo_id=mailbox_repo, filename=staging_rel, repo_type="model", token=hf_tok, local_dir=td)
        with open(enc_p, "rb") as f_enc:
            raw_b = decrypt_bytes_aes(f_enc.read(), SESSION_PASSPHRASE, aad=AAD_BLOB)
    if burn_now:
        def _bg_burn(rel_p: str, repo_p: str, tok_p: str) -> None:
            try:
                HfApi(token=tok_p).delete_file(path_in_repo=rel_p, repo_id=repo_p, repo_type="model", commit_message="burn staging")
            except Exception:
                pass
        threading.Thread(target=_bg_burn, args=(staging_rel, mailbox_repo, hf_tok), daemon=True).start()
    return Image.open(io.BytesIO(raw_b))

In [ ]:
# === Cell 2: Paste KISSSHOT_TASK_HEX (Automated v2 Lease Queue Daemon or Single-Shot Task) ===
KISSSHOT_TASK_HEX = ""

CLAIM_LEASE_TTL_SEC = 180
CLAIM_HEARTBEAT_SEC = 45
WORKER_ID = f"w-{secrets.token_hex(6)}"

def _is_queue_job_path(path: str) -> bool:
    return bool(_SAFE_QUEUE_V2_RE.fullmatch(path) or _SAFE_QUEUE_LEGACY_RE.fullmatch(path))

def _extract_job_id(queue_rel: str) -> str:
    base = os.path.basename(queue_rel)
    if base.startswith("job_") and base.endswith(".bin"):
        parts = base[:-4].split("_", 2)
        if len(parts) == 3 and parts[2]:
            return re.sub(r"[^A-Za-z0-9_-]", "", parts[2])[:48]
    return re.sub(r"[^A-Za-z0-9_-]", "", os.path.splitext(base)[0])[:48]

def _claim_path(job_id: str) -> str:
    return f"_claims/{job_id}.json"

def _receipt_path(job_id: str) -> str:
    return f"_receipts/{job_id}.json"

def _is_cas_conflict(exc: Exception) -> bool:
    resp = getattr(exc, "response", None)
    if getattr(resp, "status_code", None) in (409, 412):
        return True
    msg = str(exc).lower()
    return any(tok in msg for tok in ("412", "precondition failed", "parent_commit", "outdated", "conflict"))

def _repo_head_sha(api, repo_id: str) -> str | None:
    if not hasattr(api, "repo_info"):
        return None
    info = api.repo_info(repo_id=repo_id, repo_type="model")
    sha = getattr(info, "sha", None) if not isinstance(info, dict) else info.get("sha")
    return str(sha).strip() if sha else None

def _commit_cas(api, repo_id: str, operations: list, commit_message: str, *, parent_commit: str | None = None, max_retries: int = 3) -> dict:
    head = parent_commit or _repo_head_sha(api, repo_id)
    last_exc = None
    for attempt in range(max(1, int(max_retries))):
        try:
            kw = {
                "repo_id": repo_id,
                "repo_type": "model",
                "operations": list(operations),
                "commit_message": commit_message,
            }
            if head:
                try:
                    sig = inspect.signature(api.create_commit)
                    if "parent_commit" in sig.parameters or any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()):
                        kw["parent_commit"] = head
                except (TypeError, ValueError):
                    kw["parent_commit"] = head
            info = api.create_commit(**kw)
            oid = getattr(info, "oid", None) if not isinstance(info, dict) else info.get("oid", "")
            return {"committed": True, "commit_oid": str(oid or ""), "attempts": attempt + 1}
        except Exception as exc:
            last_exc = exc
            if not _is_cas_conflict(exc) or attempt + 1 >= max(1, int(max_retries)):
                raise
            head = _repo_head_sha(api, repo_id)
    if last_exc is not None:
        raise last_exc
    return {"committed": False, "commit_oid": "", "attempts": 0}

def _try_claim_job(
    api,
    mailbox_repo: str,
    job_id: str,
    queue_rel: str,
    *,
    worker_id: str = WORKER_ID,
    lease_ttl: int = CLAIM_LEASE_TTL_SEC,
    existing_files: set[str] | None = None,
    hf_token: str = "",
    downloader=None,
) -> dict | None:
    """K07: Optimistically acquire a lease claim at _claims/<job_id>.json using parent_commit CAS."""
    c_path = _claim_path(job_id)
    head = _repo_head_sha(api, mailbox_repo)
    files_set = existing_files if existing_files is not None else set(api.list_repo_files(repo_id=mailbox_repo, repo_type="model"))
    now = int(time.time())
    if c_path in files_set:
        try:
            dl = downloader or hf_hub_download
            with tempfile.TemporaryDirectory() as td:
                lp = dl(repo_id=mailbox_repo, filename=c_path, repo_type="model", token=hf_token or False, local_dir=td)
                with open(lp, encoding="utf-8") as cf:
                    cur_claim = json.loads(cf.read())
            exp_ts = int(cur_claim.get("expires_at", 0) or 0)
            if exp_ts > now and str(cur_claim.get("worker_id", "")) != worker_id:
                return None
        except Exception:
            return None

    claim_doc = {
        "job_id": job_id,
        "queue_rel": queue_rel,
        "worker_id": worker_id,
        "claimed_at": now,
        "heartbeat_at": now,
        "expires_at": now + int(lease_ttl),
        "status": "claimed",
    }
    raw_claim = json.dumps(claim_doc, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    try:
        _commit_cas(
            api,
            mailbox_repo,
            [CommitOperationAdd(path_in_repo=c_path, path_or_fileobj=raw_claim)],
            commit_message=f"claim lease {job_id}",
            parent_commit=head,
            max_retries=1,
        )
        return claim_doc
    except Exception as exc:
        if _is_cas_conflict(exc):
            return None
        raise

class _ClaimHeartbeat:
    """K07: Background lease renewal thread refreshing _claims/<job_id>.json until job completion."""

    def __init__(self, api, mailbox_repo: str, claim_doc: dict, *, interval_sec: float = CLAIM_HEARTBEAT_SEC, lease_ttl: int = CLAIM_LEASE_TTL_SEC) -> None:
        self._api = api
        self._repo = mailbox_repo
        self._claim = dict(claim_doc)
        self._interval = max(0.05, float(interval_sec))
        self._lease_ttl = max(5, int(lease_ttl))
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self.beats = 0

    def beat_once(self) -> bool:
        now = int(time.time())
        self._claim["heartbeat_at"] = now
        self._claim["expires_at"] = now + self._lease_ttl
        self._claim["status"] = "running"
        raw = json.dumps(self._claim, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
        try:
            _commit_cas(
                self._api,
                self._repo,
                [CommitOperationAdd(path_in_repo=_claim_path(self._claim["job_id"]), path_or_fileobj=raw)],
                commit_message=f"renew lease {self._claim['job_id']}",
                max_retries=2,
            )
            self.beats += 1
            return True
        except Exception:
            return False

    def _run(self) -> None:
        while not self._stop.wait(self._interval):
            self.beat_once()

    def start(self) -> "_ClaimHeartbeat":
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> None:
        self._stop.set()
        if self._thread is not None and self._thread.is_alive():
            self._thread.join(timeout=2.0)

def _finish_job(
    api,
    mailbox_repo: str,
    *,
    job_id: str,
    queue_rel: str,
    status: str,
    packet_rel: str = "",
    packet_bytes: bytes | None = None,
    staging_rels: list[str] | tuple[str, ...] = (),
    error: str = "",
    worker_id: str = WORKER_ID,
    max_retries: int = 3,
) -> dict:
    """K07: Atomically commit _inbox packet + _receipts/<job_id>.json and delete _queue/_claims/_staging."""
    receipt_doc = {
        "job_id": job_id,
        "queue_rel": queue_rel,
        "status": status,
        "packet_rel": packet_rel,
        "worker_id": worker_id,
        "finished_at": int(time.time()),
        "error": str(error or "")[:512],
    }
    raw_receipt = json.dumps(receipt_doc, ensure_ascii=False, separators=(",", ":")).encode("utf-8")

    def _build_ops() -> list:
        live_files = set(api.list_repo_files(repo_id=mailbox_repo, repo_type="model")) if hasattr(api, "list_repo_files") else None
        ops: list = []
        if packet_rel and packet_bytes is not None:
            ops.append(CommitOperationAdd(path_in_repo=packet_rel, path_or_fileobj=packet_bytes))
        ops.append(CommitOperationAdd(path_in_repo=_receipt_path(job_id), path_or_fileobj=raw_receipt))
        to_delete = [queue_rel, _claim_path(job_id)] + [s for s in staging_rels if s and _SAFE_STAGING_RE.fullmatch(s)]
        for rel in to_delete:
            if live_files is None or rel in live_files:
                ops.append(CommitOperationDelete(path_in_repo=rel))
        return ops

    head = _repo_head_sha(api, mailbox_repo)
    last_exc = None
    for attempt in range(max(1, int(max_retries))):
        ops = _build_ops()
        try:
            res = _commit_cas(
                api,
                mailbox_repo,
                ops,
                commit_message=f"finish kissshot job {job_id} ({status})",
                parent_commit=head,
                max_retries=1,
            )
            res["receipt_path"] = _receipt_path(job_id)
            res["packet_rel"] = packet_rel
            return res
        except Exception as exc:
            last_exc = exc
            if not _is_cas_conflict(exc) or attempt + 1 >= max(1, int(max_retries)):
                raise
            head = _repo_head_sha(api, mailbox_repo)
    if last_exc is not None:
        raise last_exc
    return {"committed": False, "receipt_path": _receipt_path(job_id), "packet_rel": packet_rel}

def _execute_media_task(
    task: dict,
    *,
    daemon_hf_token: str = "",
    daemon_mailbox_repo: str = "",
    defer_upload: bool = False,
) -> tuple[str, bytes, list[str]] | None:
    assert task.get("type") == "kissshot_media_infer", "Invalid media task payload type."
    target_repo = str(task["src_repo"])
    preset_key = str(task.get("preset_key", "preset"))
    job_arch = _job_arch(task)
    job_id = str(task.get("job_id") or "")
    variant = task.get("variant")
    skip_root = bool(task.get("skip_root", False))
    hf_token = str(task.get("hf_token") or daemon_hf_token or "")
    mailbox_repo = str(task.get("mailbox_repo") or daemon_mailbox_repo or "")
    assert hf_token and mailbox_repo, "Missing mailbox_repo or hf_token (daemon credential fallback required)."
    prompt_str = str(task["prompt"])
    neg_str = str(task.get("negative_prompt", ""))
    seed = int(task["seed"]) if int(task["seed"]) >= 0 else secrets.randbits(31)
    width, height = (int(task["width"]) // 16) * 16, (int(task["height"]) // 16) * 16
    steps, cfg_val = int(task["steps"]), float(task["cfg"])
    batch_cnt = max(1, int(task.get("batch", 1)))
    frames_req = int(task.get("frames", 81))
    strength = max(0.05, min(1.0, float(task.get("strength", 0.65))))
    upscale_2x = bool(task.get("upscale_2x", False))
    lora_repo = str(task.get("lora_repo") or "").strip()
    lora_weight = str(task.get("lora_weight") or "").strip()
    lora_strength = float(task.get("lora_strength", 0.8))
    ref_staging = str(task.get("ref_staging_blob", ""))
    mask_staging = str(task.get("mask_staging_blob", ""))
    task_revision = str(task.get("revision") or "").strip()
    del task

    is_ti2v_i2v = (job_arch == "wan_ti2v_5b" and bool(ref_staging))
    is_image_arch = job_arch in ("moss", "sdxl", "flux", "auto")
    if not is_image_arch and batch_cnt != 1:
        raise ValueError("Video architectures require batch=1.")

    api = HfApi(token=hf_token)
    # K02: Pin commit revision per repo (use registry pin, explicit revision, or freeze HEAD once)
    pinned_rev = task_revision or PINNED_MODEL_REVISIONS.get(target_repo)
    if not pinned_rev:
        try:
            pinned_rev = _repo_head_sha(HfApi(token=False), target_repo) or _repo_head_sha(api, target_repo)
        except Exception:
            pinned_rev = None

    repo_sig = hashlib.sha256(f"{target_repo}|{pinned_rev or ''}|{variant}|{skip_root}|{job_arch}|{int(is_ti2v_i2v)}".encode("utf-8")).hexdigest()
    ckpt_hash = hashlib.sha256(f"{target_repo}|{pinned_rev or ''}|{variant}|{skip_root}".encode("utf-8")).hexdigest()[:12]
    ckpt_dir = f"/tmp/kissshot_ckpt/{ckpt_hash}"
    used_staging: list[str] = [s for s in (ref_staging, mask_staging) if s]

    try:
        if _PIPE_CACHE["sig"] != repo_sig:
            if _PIPE_CACHE["pipe"] is not None:
                del _PIPE_CACHE["pipe"]
                _PIPE_CACHE["pipe"] = None
                _PIPE_CACHE["sig"] = None
                _PIPE_CACHE["lora_sig"] = None
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            if not os.path.isfile(os.path.join(ckpt_dir, ".ready")):
                os.makedirs(ckpt_dir, exist_ok=True)
                # K08: Strip all executable/script files from model repository downloads
                ignore_pats = ["*.md", "README*", ".git*"] + list(_EXECUTABLE_IGNORE_PATTERNS)
                dl_kw = {"revision": pinned_rev} if pinned_rev else {}
                if skip_root:
                    try:
                        tree = [x.path for x in HfApi(token=False).list_repo_tree(repo_id=target_repo, repo_type="model", recursive=True, **dl_kw) if hasattr(x, "size")]
                        pull_tok = False
                    except Exception:
                        tree = [x.path for x in HfApi(token=hf_token).list_repo_tree(repo_id=target_repo, repo_type="model", recursive=True, **dl_kw) if hasattr(x, "size")]
                        pull_tok = hf_token
                    allow_pats = [
                        p for p in tree
                        if not p.startswith(".git")
                        and not _is_executable_repo_file(p)
                        and not ("/" not in p and p.lower().endswith(".safetensors"))
                    ]
                    snapshot_download(repo_id=target_repo, local_dir=ckpt_dir, allow_patterns=allow_pats, token=pull_tok, **dl_kw)
                else:
                    try:
                        snapshot_download(repo_id=target_repo, local_dir=ckpt_dir, ignore_patterns=ignore_pats, token=False, **dl_kw)
                    except Exception:
                        snapshot_download(repo_id=target_repo, local_dir=ckpt_dir, ignore_patterns=ignore_pats, token=hf_token, **dl_kw)
                _strip_repo_executables(ckpt_dir)
                shutil.rmtree(os.path.join(ckpt_dir, ".cache"), ignore_errors=True)
                open(os.path.join(ckpt_dir, ".ready"), "w").write(ckpt_hash)

            if job_arch == "moss":
                from transformers import AutoTokenizer, AutoModel
                from diffusers import AutoencoderKLFlux2
                # K08: Load MOSS pipeline code ONLY from approved repository with pinned commit + SHA-256 verification
                moss_code_dir = f"/tmp/kissshot_ckpt/_approved_moss/{ckpt_hash}"
                moss_py = _approved_moss_source(target_repo, moss_code_dir, revision=pinned_rev, token=False)
                spec_m = importlib.util.spec_from_file_location("pipeline_moss_image", moss_py)
                moss_mod = importlib.util.module_from_spec(spec_m)
                sys.modules["pipeline_moss_image"] = moss_mod
                spec_m.loader.exec_module(moss_mod)
                text_enc = _load_text_encoder_vram(AutoModel, os.path.join(ckpt_dir, "text_encoder"))
                dit = moss_mod.MOSSImageTransformer.from_pretrained(
                    os.path.join(ckpt_dir, "transformer"), torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, local_files_only=True
                ).eval()
                _stream_dit_to_cuda(dit, allow_fp8_tc=True, need_shrink=(vram_gb < 18.0))
                vae = AutoencoderKLFlux2.from_pretrained(
                    os.path.join(ckpt_dir, "vae"), torch_dtype=torch.float32, low_cpu_mem_usage=True, local_files_only=True
                ).eval().to("cuda")
                sched_dir = os.path.join(ckpt_dir, "scheduler")
                moss_sched = (
                    moss_mod.MOSSImageScheduler.from_pretrained(sched_dir, local_files_only=True)
                    if hasattr(moss_mod.MOSSImageScheduler, "from_pretrained")
                    else moss_mod.MOSSImageScheduler.from_config(sched_dir, local_files_only=True)
                )
                pipe = moss_mod.MOSSImagePipeline(
                    transformer=dit,
                    scheduler=moss_sched,
                    text_encoder=text_enc,
                    tokenizer=AutoTokenizer.from_pretrained(os.path.join(ckpt_dir, "tokenizer"), local_files_only=True),
                    vae=vae,
                )
            elif job_arch == "sdxl":
                from diffusers import StableDiffusionXLPipeline
                if os.path.isfile(os.path.join(ckpt_dir, "model_index.json")):
                    kw = {"torch_dtype": torch.bfloat16, "use_safetensors": True, "local_files_only": True}
                    if variant:
                        kw["variant"] = variant
                    pipe = StableDiffusionXLPipeline.from_pretrained(ckpt_dir, **kw)
                else:
                    sf_list = sorted([os.path.join(ckpt_dir, x) for x in os.listdir(ckpt_dir) if x.lower().endswith(".safetensors")])
                    pipe = StableDiffusionXLPipeline.from_single_file(sf_list[0], torch_dtype=torch.bfloat16)
                pipe.vae.enable_tiling()
                pipe.to("cuda")
            elif job_arch == "flux":
                from diffusers import FluxPipeline, FluxTransformer2DModel
                from transformers import T5EncoderModel
                text_enc_2 = _load_text_encoder_vram(T5EncoderModel, ckpt_dir, subfolder="text_encoder_2")
                dit = FluxTransformer2DModel.from_pretrained(
                    ckpt_dir, subfolder="transformer", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, local_files_only=True
                )
                _stream_dit_to_cuda(dit, allow_fp8_tc=True, need_shrink=(vram_gb < 38.0))
                pipe = FluxPipeline.from_pretrained(
                    ckpt_dir, text_encoder_2=text_enc_2, transformer=dit, torch_dtype=torch.bfloat16, local_files_only=True
                )
                pipe.vae.enable_slicing()
                pipe.vae.enable_tiling()
                pipe.to("cuda")
            elif job_arch == "auto":
                from diffusers import DiffusionPipeline
                pipe = DiffusionPipeline.from_pretrained(ckpt_dir, torch_dtype=torch.bfloat16, local_files_only=True)
                if vram_gb >= 22.0:
                    try:
                        pipe.to("cuda")
                    except RuntimeError:
                        torch.cuda.empty_cache()
                        pipe.enable_model_cpu_offload()
                else:
                    pipe.enable_model_cpu_offload()
            elif job_arch == "cogvideox_i2v":
                from diffusers import CogVideoXImageToVideoPipeline
                from transformers import T5EncoderModel
                text_enc = _load_text_encoder_vram(T5EncoderModel, ckpt_dir, subfolder="text_encoder")
                pipe = CogVideoXImageToVideoPipeline.from_pretrained(
                    ckpt_dir, text_encoder=text_enc, torch_dtype=torch.bfloat16, local_files_only=True
                )
                pipe.vae.enable_slicing()
                pipe.vae.enable_tiling()
                if vram_gb >= 20.0:
                    pipe.to("cuda")
                else:
                    pipe.enable_model_cpu_offload()
            elif job_arch == "hunyuan15_t2v":
                from diffusers import HunyuanVideo15Pipeline
                pipe = HunyuanVideo15Pipeline.from_pretrained(ckpt_dir, torch_dtype=torch.bfloat16, local_files_only=True)
                pipe.vae.enable_tiling()
                if vram_gb >= 38.0:
                    pipe.enable_model_cpu_offload()
                else:
                    pipe.enable_sequential_cpu_offload()
            elif job_arch == "wan_t2v" or (job_arch == "wan_ti2v_5b" and not is_ti2v_i2v):
                from diffusers import WanPipeline, AutoencoderKLWan
                vae = AutoencoderKLWan.from_pretrained(ckpt_dir, subfolder="vae", torch_dtype=torch.float32, local_files_only=True)
                pipe = WanPipeline.from_pretrained(ckpt_dir, vae=vae, torch_dtype=torch.bfloat16, local_files_only=True)
                if vram_gb >= 38.0:
                    pipe.enable_model_cpu_offload()
                else:
                    pipe.enable_sequential_cpu_offload()
            elif job_arch == "wan_i2v" or is_ti2v_i2v:
                from diffusers import WanImageToVideoPipeline, AutoencoderKLWan
                vae = AutoencoderKLWan.from_pretrained(ckpt_dir, subfolder="vae", torch_dtype=torch.float32, local_files_only=True)
                pipe = WanImageToVideoPipeline.from_pretrained(ckpt_dir, vae=vae, torch_dtype=torch.bfloat16, local_files_only=True)
                if vram_gb >= 38.0:
                    pipe.enable_model_cpu_offload()
                else:
                    pipe.enable_sequential_cpu_offload()
            else:
                raise ValueError(f"Unsupported architecture: {job_arch}")

            _PIPE_CACHE["sig"] = repo_sig
            _PIPE_CACHE["pipe"] = pipe
            _PIPE_CACHE["lora_sig"] = None

        pipe = _PIPE_CACHE["pipe"]
        # K15: Apply or clear LoRA weights cleanly before generation
        lora_state = _set_lora_strength(pipe, lora_repo=lora_repo, lora_weight=lora_weight, lora_strength=lora_strength, hf_token=hf_token)
        out_blobs = []
        t0 = time.time()
        burn_staging_now = not defer_upload

        if is_image_arch:
            ref_img = _pull_staging_img(mailbox_repo, ref_staging, hf_token, burn_now=burn_staging_now).convert("RGB") if ref_staging else None
            mask_img = _pull_staging_img(mailbox_repo, mask_staging, hf_token, burn_now=burn_staging_now).convert("L") if (ref_img and mask_staging) else None
            for idx in range(batch_cnt):
                gen = torch.Generator(device="cpu").manual_seed(seed + idx)
                if ref_img is not None and job_arch == "moss":
                    img = _moss_img2img_or_inpaint(pipe, ref_img, mask_img, prompt_str, neg_str, width, height, steps, cfg_val, strength, gen)
                elif ref_img is not None:
                    from diffusers import AutoPipelineForImage2Image, AutoPipelineForInpainting
                    ref_r = ref_img.resize((width, height), Image.Resampling.LANCZOS)
                    if mask_img is not None:
                        mask_r = mask_img.resize((width, height), Image.Resampling.BILINEAR)
                        feather_m = mask_r.filter(ImageFilter.GaussianBlur(radius=6))
                        try:
                            sub_p = AutoPipelineForInpainting.from_pipe(pipe)
                            sp_sig = set(inspect.signature(sub_p.__call__).parameters.keys())
                            ikw = dict(prompt=prompt_str, image=ref_r, mask_image=mask_r, width=width, height=height,
                                       strength=strength, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                            if "negative_prompt" in sp_sig and neg_str:
                                ikw["negative_prompt"] = neg_str
                            if lora_state.get("scale_kwarg") and "cross_attention_kwargs" in sp_sig:
                                ikw["cross_attention_kwargs"] = {"scale": lora_state["strength"]}
                            img = sub_p(**ikw).images[0]
                        except Exception:
                            sub_p = AutoPipelineForImage2Image.from_pipe(pipe)
                            sp_sig = set(inspect.signature(sub_p.__call__).parameters.keys())
                            ikw = dict(prompt=prompt_str, image=ref_r, strength=strength, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                            if "negative_prompt" in sp_sig and neg_str:
                                ikw["negative_prompt"] = neg_str
                            if lora_state.get("scale_kwarg") and "cross_attention_kwargs" in sp_sig:
                                ikw["cross_attention_kwargs"] = {"scale": lora_state["strength"]}
                            img = sub_p(**ikw).images[0]
                        img = Image.composite(img.resize((width, height), Image.Resampling.LANCZOS), ref_r, feather_m)
                    else:
                        sub_p = AutoPipelineForImage2Image.from_pipe(pipe)
                        sp_sig = set(inspect.signature(sub_p.__call__).parameters.keys())
                        ikw = dict(prompt=prompt_str, image=ref_r, strength=strength, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                        if "width" in sp_sig:
                            ikw["width"], ikw["height"] = width, height
                        if "negative_prompt" in sp_sig and neg_str:
                            ikw["negative_prompt"] = neg_str
                        if lora_state.get("scale_kwarg") and "cross_attention_kwargs" in sp_sig:
                            ikw["cross_attention_kwargs"] = {"scale": lora_state["strength"]}
                        img = sub_p(**ikw).images[0]
                else:
                    sig_p = set(inspect.signature(pipe.__call__).parameters.keys())
                    kw = dict(prompt=prompt_str, width=width, height=height, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                    if "negative_prompt" in sig_p and neg_str:
                        kw["negative_prompt"] = neg_str
                        if job_arch == "moss" and "uncond_skip_layers" in sig_p:
                            kw["uncond_skip_layers"] = ()
                    if lora_state.get("scale_kwarg") and "cross_attention_kwargs" in sig_p:
                        kw["cross_attention_kwargs"] = {"scale": lora_state["strength"]}
                    img = pipe(**kw).images[0]

                if upscale_2x:
                    img = _upscale_pil_2x(img)
                mem = io.BytesIO()
                img.save(mem, format="PNG", compress_level=1)
                out_blobs.append(("png", mem.getvalue()))
        else:
            from diffusers.utils import export_to_video
            frames_cnt = ((max(17, frames_req) - 1) // 4) * 4 + 1
            gen = torch.Generator(device="cpu").manual_seed(seed)
            sig_p = set(inspect.signature(pipe.__call__).parameters.keys())
            if job_arch in ("wan_i2v", "cogvideox_i2v") or is_ti2v_i2v:
                if not ref_staging:
                    raise ValueError("Image-to-Video architecture requires an encrypted reference image.")
                ref_img = _pull_staging_img(mailbox_repo, ref_staging, hf_token, burn_now=burn_staging_now).convert("RGB")
                if job_arch == "cogvideox_i2v":
                    ref_img = ref_img.resize((width, height), Image.Resampling.LANCZOS)
                    vkw = dict(image=ref_img, prompt=prompt_str, height=height, width=width,
                               num_frames=frames_cnt, guidance_scale=cfg_val, num_inference_steps=steps, generator=gen)
                    if "negative_prompt" in sig_p and neg_str:
                        vkw["negative_prompt"] = neg_str
                    if "use_dynamic_cfg" in sig_p:
                        vkw["use_dynamic_cfg"] = True
                    vid_frames = pipe(**vkw).frames[0]
                else:
                    mod_val = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
                    max_area = width * height
                    aspect = ref_img.height / ref_img.width
                    height = max(mod_val, (round(math.sqrt(max_area * aspect)) // mod_val) * mod_val)
                    width = max(mod_val, (round(math.sqrt(max_area / aspect)) // mod_val) * mod_val)
                    ref_img = ref_img.resize((width, height), Image.Resampling.LANCZOS)
                    vid_frames = pipe(
                        image=ref_img, prompt=prompt_str, negative_prompt=neg_str, height=height, width=width,
                        num_frames=frames_cnt, guidance_scale=cfg_val, num_inference_steps=steps, generator=gen
                    ).frames[0]
            else:
                vkw = dict(prompt=prompt_str, num_frames=frames_cnt, num_inference_steps=steps, generator=gen)
                if "height" in sig_p:
                    vkw["height"], vkw["width"] = height, width
                if "guidance_scale" in sig_p:
                    vkw["guidance_scale"] = cfg_val
                if "negative_prompt" in sig_p and neg_str:
                    vkw["negative_prompt"] = neg_str
                vid_frames = pipe(**vkw).frames[0]

            shm = "/dev/shm" if os.path.isdir("/dev/shm") else tempfile.gettempdir()
            tmp_mp4 = os.path.join(shm, f"ks_{secrets.token_hex(6)}.mp4")
            out_fps = 24 if job_arch in ("hunyuan15_t2v", "wan_ti2v_5b") else 16
            try:
                export_to_video(vid_frames, tmp_mp4, fps=out_fps)
                with open(tmp_mp4, "rb") as vf:
                    out_blobs.append(("mp4", vf.read()))
            finally:
                if os.path.exists(tmp_mp4):
                    os.remove(tmp_mp4)

        is_png = bool(out_blobs and out_blobs[0][0] == "png")
        recipe_meta = {
            "job_id": job_id,
            "seed": seed,
            "model": preset_key,
            "preset": preset_key,
            "arch": job_arch,
            "prompt": prompt_str,
            "negative_prompt": neg_str,
            "base_width": width,
            "base_height": height,
            "width": width * (2 if (upscale_2x and is_png) else 1),
            "height": height * (2 if (upscale_2x and is_png) else 1),
            "steps": steps,
            "cfg": cfg_val,
            "frames": frames_req,
            "strength": strength,
            "upscale_2x": upscale_2x,
            "lora_repo": lora_repo,
            "lora_weight": lora_weight,
            "lora_strength": lora_strength if lora_repo else 0.8,
        }
        enc_zip = pack_encrypted_zip(out_blobs, SESSION_PASSPHRASE, meta=recipe_meta)
        del out_blobs
        remote_blob = f"_inbox/pkt_{int(time.time())}_{secrets.token_hex(4)}.bin"
        if defer_upload:
            print(f"✅ Done in {time.time() - t0:.1f}s (seed={seed}, arch={job_arch}). Committing atomic receipt & archive...")
            return remote_blob, enc_zip, used_staging
        if mailbox_repo not in _ENSURED_REPOS:
            api.create_repo(repo_id=mailbox_repo, repo_type="model", private=True, exist_ok=True)
            _ENSURED_REPOS.add(mailbox_repo)
        api.upload_file(path_or_fileobj=enc_zip, path_in_repo=remote_blob, repo_id=mailbox_repo, repo_type="model", commit_message="sync artifact")
        print(f"✅ Done in {time.time() - t0:.1f}s (seed={seed}, arch={job_arch}). Double-AEAD archive uploaded to {mailbox_repo}/{remote_blob}.")
        return None
    finally:
        prompt_str = ""
        neg_str = ""
        hf_token = ""
        del api

def _run_queue_daemon(cfg: dict) -> None:
    hf_tok = str(cfg["hf_token"])
    mailbox_repo = str(cfg["mailbox_repo"])
    del cfg
    api = HfApi(token=hf_tok)
    if mailbox_repo not in _ENSURED_REPOS:
        api.create_repo(repo_id=mailbox_repo, repo_type="model", private=True, exist_ok=True)
        _ENSURED_REPOS.add(mailbox_repo)
    idle_since = time.time()
    max_idle_sec = 5400
    print(f"🛰️ Kiss-Shot v2 lease queue daemon [{WORKER_ID}] listening on {mailbox_repo} (1.5s poll, 90m idle auto-stop).")
    try:
        while True:
            if time.time() - idle_since > max_idle_sec:
                print("Idle timeout (90m) reached; exiting queue daemon cleanly.")
                break
            try:
                files = api.list_repo_files(repo_id=mailbox_repo, repo_type="model")
                file_set = set(files)
                q_files = sorted([f for f in files if _is_queue_job_path(f)])
            except Exception as poll_e:
                print("Transient poll error, retrying in 3s:", str(poll_e)[:80])
                time.sleep(3)
                continue
            if not q_files:
                time.sleep(1.5)
                continue
            for qf in q_files:
                jid = _extract_job_id(qf)
                claim_doc = _try_claim_job(api, mailbox_repo, jid, qf, worker_id=WORKER_ID, existing_files=file_set, hf_token=hf_tok)
                if claim_doc is None:
                    continue
                idle_since = time.time()
                print(f"\n[Queue v2] Claimed lease {jid} ({qf}) ...")
                hb = _ClaimHeartbeat(api, mailbox_repo, claim_doc).start()
                used_staging: list[str] = []
                try:
                    shm = "/dev/shm" if os.path.isdir("/dev/shm") else None
                    with tempfile.TemporaryDirectory(dir=shm) as td:
                        lp = hf_hub_download(repo_id=mailbox_repo, filename=qf, repo_type="model", token=hf_tok, local_dir=td)
                        with open(lp, "rb") as jf:
                            job_bytes = jf.read()
                    job_obj = decrypt_queue_job(job_bytes, SESSION_PASSPHRASE)
                    job_obj.setdefault("job_id", jid)
                    used_staging = [s for s in (job_obj.get("ref_staging_blob", ""), job_obj.get("mask_staging_blob", "")) if s]
                    res = _execute_media_task(
                        job_obj,
                        daemon_hf_token=hf_tok,
                        daemon_mailbox_repo=mailbox_repo,
                        defer_upload=True,
                    )
                    hb.stop()
                    assert res is not None
                    pkt_rel, pkt_bytes, final_staging = res
                    _finish_job(
                        api,
                        mailbox_repo,
                        job_id=jid,
                        queue_rel=qf,
                        status="completed",
                        packet_rel=pkt_rel,
                        packet_bytes=pkt_bytes,
                        staging_rels=final_staging,
                        worker_id=WORKER_ID,
                    )
                    print(f"📦 Atomic commit completed: {pkt_rel} + _receipts/{jid}.json")
                except Exception as job_err:
                    hb.stop()
                    print(f"❌ Job {jid} execution error: {job_err}")
                    try:
                        _finish_job(
                            api,
                            mailbox_repo,
                            job_id=jid,
                            queue_rel=qf,
                            status="failed",
                            staging_rels=used_staging,
                            error=str(job_err),
                            worker_id=WORKER_ID,
                        )
                    except Exception as fin_err:
                        print(f"⚠️ Failed to record failure receipt for {jid}: {fin_err}")
                finally:
                    hb.stop()
                    job_obj = None
                    gc.collect()
                idle_since = time.time()
    except KeyboardInterrupt:
        print("Queue daemon stopped.")
    finally:
        hf_tok = ""
        del api

def _run_entry(raw_hex: str) -> None:
    assert raw_hex.strip(), "Paste KISSSHOT_TASK_HEX (daemon or single-shot) from your local console first."
    obj = decrypt_payload_hex(raw_hex, SESSION_PASSPHRASE)
    kind = obj.get("type")
    if kind == "kissshot_media_daemon":
        _run_queue_daemon(obj)
    elif kind == "kissshot_media_infer":
        _execute_media_task(obj)
    else:
        raise ValueError(f"Unsupported task payload type: {kind!r}")

_tmp_ks_hex = KISSSHOT_TASK_HEX
KISSSHOT_TASK_HEX = ""
scrub_ipython_history()
try:
    _run_entry(_tmp_ks_hex)
finally:
    _tmp_ks_hex = ""
    scrub_ipython_history()
    gc.collect()